# GhostID v3 — LSTM Encoder + ArcFace

Train a 128-dim behavioral embedding encoder on CMU DSL Keystroke Dynamics.

**Dataset:** `DSL-StrongPasswordData.csv` — 51 users, 20,400 sessions

**Outputs:** `ghostid_encoder.onnx`, `scaler_params.json`, analysis plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_curve, auc
import json

df = pd.read_csv('/kaggle/input/datasets/yogeshrayal/dataset/DSL-StrongPasswordData.csv')
print(f'Shape: {df.shape}, Users: {df["subject"].nunique()}')

In [ ]:
feature_cols = [c for c in df.columns if c not in ['subject', 'sessionIndex', 'rep']]
H_cols = [c for c in feature_cols if c.startswith('H.')]
for i in range(len(H_cols) - 1):
    df[f'ratio_{i}'] = df[H_cols[i]] / (df[H_cols[i + 1]] + 1e-8)
ratio_cols = [c for c in df.columns if c.startswith('ratio_')]
all_features = feature_cols + ratio_cols
print(f'Total features: {len(all_features)}')

scaler = StandardScaler()
X = scaler.fit_transform(df[all_features].values).astype(np.float32)
with open('scaler_params.json', 'w') as f:
    json.dump({'mean': scaler.mean_.tolist(), 'scale': scaler.scale_.tolist()}, f)

In [ ]:
class GhostIDEncoder(nn.Module):
    def __init__(self, input_size=41, hidden_size=256, num_layers=2, embed_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.fc = nn.Linear(hidden_size, embed_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.layer_norm(out[:, -1, :])
        out = self.dropout(out)
        out = self.fc(out)
        return nn.functional.normalize(out, p=2, dim=1)

class ArcFaceLoss(nn.Module):
    def __init__(self, embed_dim=128, num_classes=51, s=64.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embed_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m, self.sin_m = np.cos(m), np.sin(m)
        self.th = np.cos(np.pi - m)
        self.mm = np.sin(np.pi - m) * m

    def forward(self, embeddings, labels):
        cosine = nn.functional.linear(embeddings, nn.functional.normalize(self.weight))
        cosine = cosine.clamp(-1 + 1e-7, 1 - 1e-7)
        sine = torch.sqrt(1.0 - cosine ** 2)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return nn.CrossEntropyLoss()(output * self.s, labels)

print('Model classes defined')

## Training, evaluation, ONNX export

See full blueprint cells 4–12 for chronological split, ArcFace training loop,
FAR/FRR threshold analysis, EMA poisoning test, and `torch.onnx.export`.

Copy `ghostid_encoder.onnx` and `scaler_params.json` to `backend/ml/` in the repo.